# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

**Criterio de trabajo.** Se reutilizan de `Matrices.ipynb` la tokenización, la lectura
UTF-8 con alternativa Windows-1252, los identificadores y la construcción dispersa de TF.
Se conservan los mismos 100 identificadores y su orden, registrados en las salidas de ese
trabajo. Este notebook contiene las funciones necesarias y no ejecuta otros cuadernos.

Se procesa el TXT completo, incluidos encabezados y licencia, para mantener la comparación.
Se utiliza `casefold()` y el patrón Unicode `[^\W_]+`: la puntuación y el guion bajo separan
tokens; se conservan letras, tildes, ñ y números. No se eliminan stopwords ni se aplican
stemming, lematización o normalización Unicode adicional. La longitud se mide en tokens
procesados, contando repeticiones.

**Ejecución.** Se requieren Python 3.12 o posterior y `numpy pandas scipy scikit-learn
IPython ipykernel`. Se puede seleccionar el entorno `.venv-vectorsearch` del repositorio.
`RI_CORPUS` permite indicar una carpeta con los TXT o un archivo ZIP/TAR local. Si no se
configura, se revisan la ruta original, carpetas locales y el archivo de Gutenberg usado
en el deber anterior. No se descarga ni se extrae el corpus al disco.

In [1]:
from pathlib import Path, PurePosixPath
from collections import Counter
from importlib.metadata import version
import hashlib
import os
import re
import sys
import tarfile
import zipfile

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, Markdown

TOLERANCIA = 1e-10
PATRON_TOKEN = re.compile(r"[^\W_]+", flags=re.UNICODE)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.float_format", "{:.6f}".format)

def tokenizar(texto):
    return PATRON_TOKEN.findall(texto.casefold())

assert tokenizar("Tutorías, TUTORÍAS; año 2026.") == ["tutorías", "tutorías", "año", "2026"]
assert tokenizar("No requiere experiencia; a_b, C++.") == ["no", "requiere", "experiencia", "a", "b", "c"]

BASE = Path.cwd()
RAIZ = next((p for p in [BASE, *BASE.parents] if (p / "Trabajos/Matrices.ipynb").is_file()), BASE)
RUTA_ORIGINAL = Path(r"C:\Users\Jorge\Documents\EPN-7mo Semestre\Recuperacion Informacion\epub")
RUTA_ARCHIVO = Path(os.getenv("LOCALAPPDATA", str(Path.home() / ".local"))) / "GutenbergRI/txt-files.tar.zip"
ruta_configurada = os.getenv("RI_CORPUS")
candidatas = ([Path(ruta_configurada)] if ruta_configurada else
              [RUTA_ORIGINAL, BASE / "epub", BASE / "gutenberg_grande",
               RAIZ / "epub", RAIZ / "gutenberg_grande", RUTA_ARCHIVO])
RUTA_CORPUS = next((p for p in candidatas if p.exists()), None)
if RUTA_CORPUS is None:
    raise FileNotFoundError("No se encontro el corpus. Configure RI_CORPUS con la carpeta TXT o el ZIP/TAR local. "
                            + "Rutas revisadas: " + "; ".join(map(str, candidatas)))

print(f"Existe la ruta original: {RUTA_ORIGINAL.is_dir()}")
print(f"Fuente local seleccionada: {RUTA_CORPUS}")
display(pd.DataFrame({"Dependencia": ["Python", "numpy", "pandas", "scipy", "scikit-learn"],
                      "Versión": [sys.version.split()[0], *[version(p) for p in
                                  ["numpy", "pandas", "scipy", "scikit-learn"]]]}))


Existe la ruta original: False
Fuente local seleccionada: C:\Users\jorge\AppData\Local\GutenbergRI\txt-files.tar.zip


,Dependencia,Versión
0,Python,3.12.14
1,numpy,2.5.4
2,pandas,3.1.1
3,scipy,1.18.1
4,scikit-learn,1.9.1


In [2]:
IDS_ORIGINALES = """
1 2 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 100 101 102
103 104 105 106 107 108 109 110 111 112 113 115 117 118 119 120 121 122 123 124
125 126 128 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146
147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166
167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 200 201 202 203 204
""".split()

IDS_ELEGIDOS = set(IDS_ORIGINALES)

def identificar_documento(nombre):
    archivo = PurePosixPath(str(nombre).replace("\\", "/")).name
    coincidencia = re.fullmatch(r"pg(\d+)\.txt", archivo, flags=re.IGNORECASE)
    return coincidencia[1] if coincidencia and coincidencia[1] in IDS_ELEGIDOS else None

def leer_tar(flujo):
    with tarfile.open(fileobj=flujo, mode="r|*") as archivo_tar:
        for miembro in archivo_tar:
            if miembro.isfile() and identificar_documento(miembro.name):
                with archivo_tar.extractfile(miembro) as documento:
                    yield miembro.name, documento.read()

def iterar_documentos(ruta):
    # Se leen unicamente los TXT seleccionados, sin extraer rutas del archivo.
    if ruta.is_dir():
        archivos = sorted(ruta.rglob("*.txt"), key=lambda p: (
            int(p.parent.name) if p.parent.name.isdigit() else float("inf"),
            str(p.relative_to(ruta)).casefold()))
        for archivo in archivos:
            if identificar_documento(archivo.name):
                yield str(archivo.relative_to(ruta)), archivo.read_bytes()
    elif zipfile.is_zipfile(ruta):
        with zipfile.ZipFile(ruta) as archivo_zip:
            for miembro in sorted(archivo_zip.infolist(), key=lambda m: m.filename):
                if miembro.filename.lower().endswith((".tar", ".tar.gz", ".tgz")):
                    with archivo_zip.open(miembro) as flujo:
                        yield from leer_tar(flujo)
                elif not miembro.is_dir() and identificar_documento(miembro.filename):
                    yield miembro.filename, archivo_zip.read(miembro)
    else:
        with ruta.open("rb") as flujo:
            yield from leer_tar(flujo)

def leer_texto(contenido):
    try:
        texto = contenido.decode("utf-8-sig")
        codificacion = "utf-8-sig"
    except UnicodeDecodeError:
        texto = contenido.decode("cp1252")
        codificacion = "cp1252"
    # Se reproducen los saltos de linea de Path.read_text() del trabajo anterior.
    return texto.replace("\r\n", "\n").replace("\r", "\n"), codificacion

def obtener_titulo(texto, doc_id):
    encabezado = re.split(r"\*{3}\s*START OF", texto[:20000], maxsplit=1, flags=re.I)[0]
    patron = r"^Title:[ \t]*([^\n]*(?:\n(?!\s*\n|[ \t]*[A-Za-z][A-Za-z -]*:|\*{3})[^\n]+)*)"
    coincidencia = re.search(patron, encabezado, flags=re.I | re.M)
    return " ".join(coincidencia[1].split()) if coincidencia else f"Gutenberg {doc_id} (pg{doc_id}.txt)"


In [3]:
documentos_por_id = {}
huellas = set()
candidatos = iterar_documentos(RUTA_CORPUS)
try:
    for nombre, contenido in candidatos:
        doc_id = identificar_documento(nombre)
        if doc_id in documentos_por_id:
            continue
        texto, codificacion = leer_texto(contenido)
        if "project gutenberg" not in texto[:5000].casefold():
            continue
        huella = hashlib.sha256(texto.encode("utf-8")).hexdigest()
        tokens = [sys.intern(t) for t in tokenizar(texto)]
        if not tokens or huella in huellas:
            continue
        documentos_por_id[doc_id] = {
            "id": doc_id, "titulo": obtener_titulo(texto, doc_id), "texto": texto,
            "tokens": tokens, "longitud": len(tokens), "unicos": len(set(tokens)),
            "archivo": nombre, "codificacion": codificacion, "sha256": huella}
        huellas.add(huella)
        if len(documentos_por_id) == len(IDS_ORIGINALES):
            break
finally:
    candidatos.close()

faltantes = [doc_id for doc_id in IDS_ORIGINALES if doc_id not in documentos_por_id]
if faltantes:
    raise ValueError(f"Faltan los TXT originales de estos IDs: {faltantes}. Revise RI_CORPUS.")
documentos = [documentos_por_id[doc_id] for doc_id in IDS_ORIGINALES]
ids = [doc["id"] for doc in documentos]
N = len(documentos)
longitudes = np.array([doc["longitud"] for doc in documentos], dtype=np.int64)
vocabulario_corpus = set(t for doc in documentos for t in doc["tokens"])
assert N == 100 and len(set(ids)) == N and len(huellas) == N
assert ids == IDS_ORIGINALES

tabla_corpus = pd.DataFrame({"Identificador": ids, "Título": [d["titulo"] for d in documentos],
                             "Número de términos únicos": [d["unicos"] for d in documentos]})
with pd.option_context("display.max_rows", N):
    display(tabla_corpus)
display(pd.DataFrame({"Estadística": ["Documentos", "Vocabulario", "Tokens totales",
                                     "Longitud mínima (tokens)", "Longitud media (tokens)",
                                     "Longitud máxima (tokens)"],
                      "Valor": [N, len(vocabulario_corpus), longitudes.sum(),
                                longitudes.min(), longitudes.mean(), longitudes.max()]}))
# Se contrastan los totales registrados en Matrices.ipynb.
assert longitudes.sum() == 13_156_915 and len(vocabulario_corpus) == 191_063
print("Se recuperaron los mismos 100 IDs y los totales del trabajo anterior.")


,Identificador,Título,Número de términos únicos
0,1,The Declaration of Independence of the United States of America,1212
1,2,The United States Bill of Rights,844
2,10,The King James Version of the Bible,13061
3,11,Alice's Adventures in Wonderland,3031
4,12,Through the Looking-Glass,3284
5,13,The Hunting of the Snark: An Agony in Eight Fits,1978
6,14,The 1990 CIA World Factbook,18882
7,15,"Moby-Dick; or, The Whale",17347
8,16,Peter Pan [Peter and Wendy],5218
9,17,"The Book of Mormon an account written by the hand of Mormon, upon plates taken from the plates of Nephi",6004


,Estadística,Valor
0,Documentos,100.000000
1,Vocabulario,191063.000000
2,Tokens totales,13156915.000000
3,Longitud mínima (tokens),3304.000000
4,Longitud media (tokens),131569.150000
5,Longitud máxima (tokens),1363417.000000


Se recuperaron los mismos 100 IDs y los totales del trabajo anterior.


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

Se reutiliza `Counter` para TF y se cuentan las claves de cada documento para DF. El IDF emplea exclusivamente `numpy.log(N / DF) + 1`; los pesos no se normalizan. Se eligen seis pares presentes en los documentos reales.

In [4]:
def calcular_tf(tokens):
    return Counter(tokens)

def calcular_df(frecuencias_documentos):
    return Counter(termino for frecuencias in frecuencias_documentos for termino in frecuencias)

def calcular_idf(frecuencias_documentales, numero_documentos):
    valores_df = np.asarray(frecuencias_documentales, dtype=float)
    if np.any((valores_df <= 0) | (valores_df > numero_documentos)):
        raise ValueError("DF debe estar entre 1 y N para los terminos del vocabulario.")
    return np.log(numero_documentos / valores_df) + 1

def calcular_peso_tfidf(frecuencia, valor_idf):
    return frecuencia * valor_idf

contadores = [calcular_tf(doc["tokens"]) for doc in documentos]
df_por_termino = calcular_df(contadores)
pares_ejemplo = [("1", "independence"), ("2", "rights"), ("11", "alice"),
                 ("11", "rabbit"), ("120", "treasure"), ("161", "marianne")]
ejemplos = []
for doc_id, termino in pares_ejemplo:
    frecuencia = contadores[ids.index(doc_id)][termino]
    frecuencia_documental = df_por_termino[termino]
    assert frecuencia > 0
    valor_idf = float(calcular_idf(frecuencia_documental, N))
    ejemplos.append({"Documento": doc_id, "Término": termino, "TF": frecuencia,
                     "DF": frecuencia_documental, "IDF": valor_idf,
                     "TF-IDF": calcular_peso_tfidf(frecuencia, valor_idf)})
display(pd.DataFrame(ejemplos))


,Documento,Término,TF,DF,IDF,TF-IDF
0,1,independence,8,50,1.693147,13.545177
1,2,rights,6,62,1.478036,8.868215
2,11,alice,403,21,2.560648,1031.941043
3,11,rabbit,53,25,2.386294,126.473601
4,120,treasure,67,53,1.634878,109.536844
5,161,marianne,566,4,4.218876,2387.883717


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [5]:
def construir_matriz_tf(frecuencias_documentos, columnas):
    filas, indices, valores = [], [], []
    for fila, frecuencias in enumerate(frecuencias_documentos):
        for termino, frecuencia in frecuencias.items():
            if termino in columnas:
                filas.append(fila)
                indices.append(columnas[termino])
                valores.append(frecuencia)
    matriz = csr_matrix((valores, (filas, indices)),
                        shape=(len(frecuencias_documentos), len(columnas)), dtype=np.int64)
    matriz.sort_indices()
    return matriz

vocabulario = sorted(vocabulario_corpus)
columnas = {termino: i for i, termino in enumerate(vocabulario)}
matriz_tf = construir_matriz_tf(contadores, columnas)
df = np.asarray((matriz_tf > 0).sum(axis=0)).ravel()
idf = calcular_idf(df, N)
matriz_tfidf = matriz_tf.multiply(idf).tocsr()
matriz_tfidf.sort_indices()

display(pd.DataFrame({"Representación": ["TF", "TF-IDF"],
                      "Dimensiones": [matriz_tf.shape, matriz_tfidf.shape],
                      "Elementos distintos de cero": [matriz_tf.nnz, matriz_tfidf.nnz],
                      "Vocabulario": [len(vocabulario)] * 2}))
# Se muestran palabras de las consultas y pesos de seis documentos identificables.
terminos_vista = ["alice", "rabbit", "wonderland", "treasure", "island", "pirates",
                  "love", "marriage", "elinor", "marianne", "independence", "constitution"]
columnas_vista = [columnas[t] for t in terminos_vista]
filas_vista = [ids.index(doc_id) for doc_id in ["1", "2", "11", "120", "161", "100"]]
vista_tfidf = pd.DataFrame(matriz_tfidf[filas_vista][:, columnas_vista].toarray(),
                           index=[ids[i] for i in filas_vista], columns=terminos_vista)
vista_tfidf.index.name = "Identificador"
display(vista_tfidf.round(4))


,Representación,Dimensiones,Elementos distintos de cero,Vocabulario
0,TF,"(100, 191063)",881166,191063
1,TF-IDF,"(100, 191063)",881166,191063


,alice,rabbit,wonderland,treasure,island,pirates,love,marriage,elinor,marianne,independence,constitution
Identificador,,,,,,,,,,,,
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,13.545200,1.798500
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.395500
11,1031.941000,126.473600,23.118100,0.000000,0.000000,0.000000,3.383500,0.000000,0.000000,0.000000,0.000000,0.000000
120,0.000000,0.000000,0.000000,109.536800,131.498100,45.232400,1.127800,0.000000,0.000000,0.000000,0.000000,0.000000
161,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,86.843200,64.329600,3364.735800,2387.883700,11.852000,5.395500
100,79.380100,9.545200,0.000000,106.267100,61.266100,39.911000,2780.109300,190.064600,0.000000,0.000000,0.000000,3.597000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [6]:
def analizar_tokens(tokens):
    return tokens

def maxima_diferencia_dispersa(a, b):
    assert a.shape == b.shape
    diferencia = (a - b).tocsr()
    return float(np.max(np.abs(diferencia.data), initial=0.0))

# El analizador recibe las listas existentes y conserva todos sus tokens.
vectorizador = TfidfVectorizer(analyzer=analizar_tokens, vocabulary=columnas,
                              lowercase=False, token_pattern=None,
                              use_idf=True, smooth_idf=False, sublinear_tf=False,
                              norm=None, dtype=np.float64)
tokens_documentos = [doc["tokens"] for doc in documentos]
matriz_sklearn = vectorizador.fit_transform(tokens_documentos).tocsr()
assert vectorizador.get_feature_names_out().tolist() == vocabulario
diferencia_tfidf = maxima_diferencia_dispersa(matriz_tfidf, matriz_sklearn)
assert diferencia_tfidf <= TOLERANCIA
assert np.allclose(idf, vectorizador.idf_, atol=TOLERANCIA, rtol=0)

display(pd.DataFrame({"Matriz": ["Manual", "scikit-learn"],
                      "Dimensiones": [matriz_tfidf.shape, matriz_sklearn.shape]}))
print(f"Máxima diferencia absoluta: {diferencia_tfidf:.12g}")
print(f"Verificación con tolerancia {TOLERANCIA:g}: {diferencia_tfidf <= TOLERANCIA}")


,Matriz,Dimensiones
0,Manual,"(100, 191063)"
1,scikit-learn,"(100, 191063)"


Máxima diferencia absoluta: 0
Verificación con tolerancia 1e-10: True


La comparación recorre la diferencia de las matrices completas en formato disperso. El vocabulario fijo garantiza que cada columna representa el mismo término en ambas versiones; el analizador personalizado evita el preprocesamiento predeterminado.

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

Las cuatro consultas se formulan en inglés porque buscan contenido de obras del corpus en ese idioma. Sus referencias son los documentos 11 (*Alice’s Adventures in Wonderland*), 120 (*Treasure Island*), 161 (*Sense and Sensibility*) y 1–2 (documentos históricos de Estados Unidos). Los términos ajenos al vocabulario se ignoran y no modifican los IDF.

In [7]:
consultas = [
    {"id": "Q1", "texto": "alice rabbit wonderland", "referencia": "11",
     "necesidad": "Encontrar el relato fantástico de Alicia y el conejo en Wonderland."},
    {"id": "Q2", "texto": "pirates treasure island", "referencia": "120",
     "necesidad": "Encontrar aventuras sobre piratas y la búsqueda de un tesoro en una isla."},
    {"id": "Q3", "texto": "love marriage elinor marianne", "referencia": "161",
     "necesidad": "Encontrar conflictos amorosos y matrimoniales de Elinor y Marianne."},
    {"id": "Q4", "texto": "united states independence constitution", "referencia": "1",
     "necesidad": "Encontrar documentos sobre independencia, gobierno y derechos de Estados Unidos."},
]
for consulta in consultas:
    consulta["tokens"] = tokenizar(consulta["texto"])
    consulta["fuera_vocabulario"] = [t for t in consulta["tokens"] if t not in columnas]
    referencia = contadores[ids.index(consulta["referencia"])]
    assert any(t in referencia for t in consulta["tokens"])
display(pd.DataFrame({"Consulta": [q["id"] for q in consultas],
                      "Texto": [q["texto"] for q in consultas],
                      "Necesidad de información": [q["necesidad"] for q in consultas],
                      "Términos procesados": [q["tokens"] for q in consultas],
                      "Fuera del vocabulario": [q["fuera_vocabulario"] for q in consultas]}))

idf_antes_consultas = idf.copy()
idf_sklearn_antes = vectorizador.idf_.copy()
vocabulario_antes = vectorizador.vocabulary_.copy()
tokens_consultas = [q["tokens"] for q in consultas]
tf_consultas = construir_matriz_tf([calcular_tf(tokens) for tokens in tokens_consultas], columnas)
consultas_manual = tf_consultas.multiply(idf).tocsr()
consultas_sklearn = vectorizador.transform(tokens_consultas).tocsr()
diferencia_consultas = maxima_diferencia_dispersa(consultas_manual, consultas_sklearn)
assert diferencia_consultas <= TOLERANCIA
assert np.array_equal(idf, idf_antes_consultas)
assert np.array_equal(vectorizador.idf_, idf_sklearn_antes)
assert vectorizador.vocabulary_ == vocabulario_antes
print(f"Vectores de consulta: {consultas_manual.shape}; máxima diferencia: {diferencia_consultas:.12g}")


,Consulta,Texto,Necesidad de información,Términos procesados,Fuera del vocabulario
0,Q1,alice rabbit wonderland,Encontrar el relato fantástico de Alicia y el conejo en Wonderland.,"[alice, rabbit, wonderland]",[]
1,Q2,pirates treasure island,Encontrar aventuras sobre piratas y la búsqueda de un tesoro en una isla.,"[pirates, treasure, island]",[]
2,Q3,love marriage elinor marianne,Encontrar conflictos amorosos y matrimoniales de Elinor y Marianne.,"[love, marriage, elinor, marianne]",[]
3,Q4,united states independence constitution,"Encontrar documentos sobre independencia, gobierno y derechos de Estados Unidos.","[united, states, independence, constitution]",[]


Vectores de consulta: (4, 191063); máxima diferencia: 0


In [8]:
def coseno_explicito(consultas_vector, documentos_vector):
    producto_punto = (consultas_vector @ documentos_vector.T).toarray()
    normas_consultas = np.sqrt(np.asarray(consultas_vector.multiply(consultas_vector).sum(axis=1)).ravel())
    normas_documentos = np.sqrt(np.asarray(documentos_vector.multiply(documentos_vector).sum(axis=1)).ravel())
    denominador = normas_consultas[:, None] * normas_documentos[None, :]
    similitudes = np.divide(producto_punto, denominador, out=np.zeros_like(producto_punto),
                            where=denominador > 0)
    return similitudes, producto_punto, normas_consultas, normas_documentos

similitudes_manual, productos_punto, normas_q, normas_d = coseno_explicito(consultas_manual, matriz_tfidf)
similitudes_sklearn = cosine_similarity(consultas_sklearn, matriz_sklearn)
diferencia_coseno = float(np.max(np.abs(similitudes_manual - similitudes_sklearn)))
assert np.allclose(similitudes_manual, similitudes_sklearn, atol=TOLERANCIA, rtol=0)

# Se comprueban consultas y documentos de norma cero.
vector_cero = csr_matrix((1, len(vocabulario)), dtype=float)
assert np.array_equal(coseno_explicito(vector_cero, matriz_tfidf)[0], np.zeros((1, N)))
assert np.array_equal(coseno_explicito(consultas_manual, vector_cero)[0], np.zeros((4, 1)))
assert np.array_equal(cosine_similarity(vector_cero, matriz_tfidf), np.zeros((1, N)))
assert np.array_equal(cosine_similarity(consultas_manual, vector_cero), np.zeros((4, 1)))
assert vectorizador.transform([[]]).nnz == 0
termino_ausente = "xyzterminoinexistente987654321"
while termino_ausente in columnas:
    termino_ausente += "x"
assert vectorizador.transform([tokenizar(termino_ausente)]).nnz == 0

print(f"Similitudes verificadas: {similitudes_manual.size} (4 consultas × {N} documentos)")
print(f"Máxima diferencia de coseno: {diferencia_coseno:.12g}")
print("Los vectores de norma cero producen similitud cero.")


Similitudes verificadas: 400 (4 consultas × 100 documentos)
Máxima diferencia de coseno: 1.17267306976e-15
Los vectores de norma cero producen similitud cero.


In [9]:
q_ejemplo, d_ejemplo = 0, ids.index("11")
productos_compartidos = consultas_manual[q_ejemplo].multiply(matriz_tfidf[d_ejemplo]).tocsr()
productos_compartidos.eliminate_zeros()
indices_compartidos = productos_compartidos.indices
detalle_coseno = pd.DataFrame({
    "Términos compartidos": [vocabulario[j] for j in indices_compartidos],
    "Peso consulta": [float(consultas_manual[q_ejemplo, j]) for j in indices_compartidos],
    "Peso documento": [float(matriz_tfidf[d_ejemplo, j]) for j in indices_compartidos],
    "Productos parciales": productos_compartidos.data,
    "Producto punto": productos_punto[q_ejemplo, d_ejemplo],
    "Norma consulta": normas_q[q_ejemplo], "Norma documento": normas_d[d_ejemplo],
    "Similitud": similitudes_manual[q_ejemplo, d_ejemplo]})
assert np.isclose(productos_compartidos.data.sum(), productos_punto[q_ejemplo, d_ejemplo],
                  atol=TOLERANCIA, rtol=0)
display(Markdown(f"**Comprobación individual:** Q1 y documento {ids[d_ejemplo]} — {documentos[d_ejemplo]['titulo']}."))
display(detalle_coseno)
display(Markdown("Se multiplican los pesos de cada término compartido y se suman los productos parciales. "
                 "El producto punto se divide entre las dos normas. La norma del documento incluye "
                 "todos sus términos, incluso aquellos que no aparecen en la consulta."))


**Comprobación individual:** Q1 y documento 11 — Alice's Adventures in Wonderland.

,Términos compartidos,Peso consulta,Peso documento,Productos parciales,Producto punto,Norma consulta,Norma documento,Similitud
0,alice,2.560648,1031.941043,2642.437507,3020.590226,4.812316,3248.385729,0.193228
1,rabbit,2.386294,126.473601,301.803241,3020.590226,4.812316,3248.385729,0.193228
2,wonderland,3.302585,23.118096,76.349478,3020.590226,4.812316,3248.385729,0.193228


Se multiplican los pesos de cada término compartido y se suman los productos parciales. El producto punto se divide entre las dos normas. La norma del documento incluye todos sus términos, incluso aquellos que no aparecen en la consulta.

### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

Se ordena por puntuación descendente y, en empates exactos, por identificador numérico ascendente. Las puntuaciones conservan su precisión interna. Cada fragmento es una subcadena del texto original de hasta 180 caracteres: se busca la primera coincidencia de un término de consulta en el cuerpo de la obra. Si no se encuentra, se utiliza el inicio del cuerpo y se informa el identificador correspondiente. Los encabezados y la licencia se excluyen de la selección de fragmentos cuando existen delimitadores.

In [10]:
def ordenar_documentos(puntuaciones):
    return sorted(range(N), key=lambda i: (-float(puntuaciones[i]), int(ids[i])))

def extraer_fragmento(texto, terminos, limite=180):
    inicio_marca = re.search(r"^\*{3}\s*START OF (?:THE|THIS) PROJECT GUTENBERG[^\n]*", texto, re.I | re.M)
    fin_marca = re.search(r"^\*{3}\s*END OF (?:THE|THIS) PROJECT GUTENBERG", texto, re.I | re.M)
    inicio_cuerpo = inicio_marca.end() if inicio_marca else 0
    fin_cuerpo = fin_marca.start() if fin_marca and fin_marca.start() > inicio_cuerpo else len(texto)
    buscados = set(terminos)
    coincidencia = next((m for m in PATRON_TOKEN.finditer(texto, inicio_cuerpo, fin_cuerpo)
                         if m[0].casefold() in buscados), None)
    criterio = "Coincidencia con la consulta" if coincidencia else "Inicio del cuerpo (sin coincidencia)"
    inicio = max(inicio_cuerpo, coincidencia.start() - 55) if coincidencia else inicio_cuerpo
    while inicio < fin_cuerpo and texto[inicio].isspace():
        inicio += 1
    fin = min(inicio + limite, fin_cuerpo)
    return {"fragmento": texto[inicio:fin], "inicio": inicio, "fin": fin, "criterio": criterio}

fragmentos_guardados = {}

def construir_tabla_ranking(puntuaciones, indice_consulta):
    registros = []
    for posicion, fila in enumerate(ordenar_documentos(puntuaciones)[:5], start=1):
        clave = (indice_consulta, fila)
        if clave not in fragmentos_guardados:
            fragmentos_guardados[clave] = extraer_fragmento(documentos[fila]["texto"],
                                                          consultas[indice_consulta]["tokens"])
        datos = fragmentos_guardados[clave]
        registros.append({"Posición": posicion, "Identificador": ids[fila],
                          "Título": documentos[fila]["titulo"], "Puntuación": float(puntuaciones[fila]),
                          "Fragmento": datos["fragmento"]})
    return pd.DataFrame(registros)

rankings_tfidf = []
for qi, consulta in enumerate(consultas):
    orden_manual = ordenar_documentos(similitudes_manual[qi])
    orden_sklearn = ordenar_documentos(similitudes_sklearn[qi])
    assert orden_manual == orden_sklearn
    assert np.allclose(similitudes_manual[qi, orden_manual], similitudes_sklearn[qi, orden_sklearn],
                       atol=TOLERANCIA, rtol=0)
    tabla = construir_tabla_ranking(similitudes_manual[qi], qi)
    rankings_tfidf.append(tabla)
    display(Markdown(f"**{consulta['id']}: {consulta['texto']} — TF-IDF / coseno**"))
    display(tabla)

sin_coincidencia = [(consultas[qi]["id"], ids[fila]) for (qi, fila), datos in fragmentos_guardados.items()
                    if datos["criterio"].startswith("Inicio")]
if sin_coincidencia:
    print("Fragmentos tomados del inicio del cuerpo (consulta, ID):", sin_coincidencia)
print("Los cuatro rankings completos y sus top-5 coinciden con scikit-learn.")


**Q1: alice rabbit wonderland — TF-IDF / coseno**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,11,Alice's Adventures in Wonderland,0.193228,[Illustration]\n\n\n\n\nAlice’s Adventures in Wonderland\n\nby Lewis Carroll\n\nTHE MILLENNIUM FULCRUM EDITION 3.0\n\nContents\n\n CHAPTER I. Down the Rabbit-Hole\n CHAPTER II. The Pool o
1,2,12,Through the Looking-Glass,0.183056,[Illustration]\n\n\n\n\nTHROUGH THE LOOKING-GLASS\n\nAnd What Alice Found There\n\nBy Lewis Carroll\n\nThe Millennium Fulcrum Edition 1.7\n\n\n\n\nDRAMATIS PERSONÆ.\n(_As arranged before commenceme
2,3,133,The Damnation of Theron Ware,0.019309,"patted her\nhand.\n\n“Somebody must have the poor places, Alice,” he said consolingly. “I am\na young man yet, remember. We must take our turn, and be patient. For\n'we know that all th"
3,4,159,The island of Doctor Moreau,0.004332,"eled truck. The llama was still\non the launch with the rabbit hutches; the staghounds were still lashed\nto the thwarts. The pile of things completed, all three men laid hold\nof the"
4,5,19,The Song of Hiawatha,0.002713,"rom the kingdom of Wabasso,\nFrom the land of the White Rabbit.\n\nHe had stolen the Belt of Wampum\nFrom the neck of Mishe-Mokwa,\nFrom the Great Bear of the mountains,\nFrom the terror"


**Q2: pirates treasure island — TF-IDF / coseno**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,16,Peter Pan [Peter and Wendy],0.026779,"AY, COME AWAY!\n Chapter IV. THE FLIGHT\n Chapter V. THE ISLAND COME TRUE\n Chapter VI. THE LITTLE HOUSE\n Chapter VII. THE HOME UNDER THE GROUND\n Chapter VIII. THE MERMAIDS’ LAGOON\n C"
1,2,120,Treasure Island,0.018740,"TREASURE ISLAND\n\nby Robert Louis Stevenson\n\n\n\n\nTREASURE ISLAND\n\nTo S.L.O., an American gentleman in accordance with whose classic taste\nthe following narrative has been designed, i"
2,3,159,The island of Doctor Moreau,0.008599,[Illustration]\n\n\n\n\nThe Island of Doctor Moreau\n\nby H. G. Wells\n\n\nContents\n\n INTRODUCTION\n I. IN THE DINGEY OF THE “LADY VAIN”\n II. THE MAN WHO WAS GOING NOWHERE\n III. THE STRANGE F
3,4,180,The 1994 CIA World Factbook,0.007441,"ralia\nAustria\nAzerbaijan\n\nB\nBahamas, The\nBahrain\nBaker Island\nBangladesh\nBarbados\nBassas da India\nBelarus\nBelgium\nBelize\nBenin\nBermuda\nBhutan\nBolivia\nBosnia and Herzegovina\nBotswan"
4,5,14,The 1990 CIA World Factbook,0.006989,"ia\n Austria\n\n Bahamas, The\n Bahrain\n Baker Island\n Bangladesh\n Barbados\n Bassas da India\n Belgium\n Belize\n Benin\n Bermuda\n Bhutan\n Bolivia\n"


**Q3: love marriage elinor marianne — TF-IDF / coseno**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,161,Sense and Sensibility,0.327458,"children added a relish to his existence.\n\nBy a former marriage, Mr. Henry Dashwood had one son: by his present\nlady, three daughters. The son, a steady respectable young man, was\n"
1,2,100,The Complete Works of William Shakespeare,0.006337,GEDY OF JULIUS CAESAR\n THE TRAGEDY OF KING LEAR\n LOVE’S LABOUR’S LOST\n THE TRAGEDY OF MACBETH\n MEASURE FOR MEASURE\n THE MERCHANT OF VENICE\n THE MERRY WIVES OF WIN
2,3,138,George Sand: Some Aspects of Her Life and Writings,0.006241,(DE BOURGES)\n VI A CASE OF MATERNAL AFFECTION IN LOVE\n VII THE HUMANITARIAN DREAM\n VIII 1848\n IX THE 'BONNE DAME' OF NOHANT\n X THE GENIUS OF THE WRITER\n
3,4,163,Flower Fables,0.005089,"nts\n\n FLOWER FABLES\n The Frost King: or, The Power of Love\n Eva’s Visit to Fairy-Land\n The Flower’s Lesson\n Lily-Bell and Thistledown\n Little Bud\n Clover-Blossom\n Little Annie’s D"
4,5,145,Middlemarch,0.003787,XXI.\n CHAPTER XXXII.\n CHAPTER XXXIII.\n\n BOOK IV. THREE LOVE PROBLEMS.\n CHAPTER XXXIV.\n CHAPTER XXXV.\n CHAPTER XXXVI.\n CHAPTER XXXVII.\n CHAPTER XXXVIII.\n CHAPTER XXXIX.\n CHAPTER XL.


**Q4: united states independence constitution — TF-IDF / coseno**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,1,The Declaration of Independence of the United States of America,0.055538,"The United States Declaration of Independence was the first E-text\nreleased by Project Gutenberg, early in 1971. The title was stored\nin an emailed instruction set which required a"
1,2,2,The United States Bill of Rights,0.053863,"The United States Bill of Rights.\n\nThe Ten Original Amendments to the Constitution of the United States\nPassed by Congress September 25, 1789\nRatified December 15, 1791\n\n\n\n\nI\n\nCong"
2,3,18,The Federalist Papers,0.036326,o. VI. Concerning Dangers from Dissensions Between the States\n FEDERALIST No. VII. The Same Subject Continued (Concerning Dangers from Dissensions Between the States)\n FEDERALIST N
3,4,117,Symphony No. 5 in C minor Opus 67,0.033849,"Beethoven's Fifth Symphony, in C-minor, Opus #67\n\n\nby Ludwig van Beethoven\n\n\n\n(C) 1994, 2001 Geof Pawlicki\n(P) 1994, 2001 Geof Pawlicki\n\n\nThe accompanying files contain the four mo"
4,5,156,"Symphony No. 5 in C minor, Opus 67",0.033801,"Produced by Geof Pawlicki\n\n\n\n\n\n\n\n\nBeethoven's Fifth Symphony, in C-minor, Opus #67\n\n\nby Ludwig van Beethoven\n\n\n\n(C) 1994, 2001 Geof Pawlicki\n(P) 1994, 2001 Geof Pawlicki\n\n\nThe ..."


Fragmentos tomados del inicio del cuerpo (consulta, ID): [('Q4', '117'), ('Q4', '156')]
Los cuatro rankings completos y sus top-5 coinciden con scikit-learn.


### Complemento: BM25 y comparación de rankings

Se añade BM25 para completar el objetivo general, conservando las seis actividades originales.
Se emplean $k_1=1.5$ y $b=0.75$:

$$\operatorname{BM25}(q,d)=\sum_{t\in q\cap V}
\operatorname{IDF}_{BM25}(t)\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1(1-b+b\,|d|/\overline{|d|})}$$

$$\operatorname{IDF}_{BM25}(t)=\ln\left(1+\frac{N-DF(t)+0.5}{DF(t)+0.5}\right)$$

Esta variante positiva del IDF de Robertson se calcula con `numpy.log1p`; es distinta de
$\ln(N/DF)+1$ usado en TF-IDF. $k_1$ controla la saturación de TF y $b$ la corrección por
longitud. $|d|$ es la cantidad real de tokens y $\overline{|d|}$ su media en los mismos
100 documentos. Cada término distinto de la consulta aporta una vez; no se aplica un
factor adicional por repeticiones en la consulta. Los términos fuera de $V$ se ignoran.

In [11]:
K1, B = 1.5, 0.75
longitud_media = float(longitudes.mean())
idf_bm25 = np.log1p((N - df + 0.5) / (df + 0.5))
tf_por_columnas = matriz_tf.tocsc()

def calcular_bm25(tokens_consulta, k1=K1, b=B):
    normalizador = k1 * (1 - b + b * longitudes / longitud_media)
    puntuaciones = np.zeros(N, dtype=float)
    indices = sorted({columnas[t] for t in tokens_consulta if t in columnas})
    for j in indices:
        frecuencias = tf_por_columnas[:, j].toarray().ravel()
        puntuaciones += idf_bm25[j] * frecuencias * (k1 + 1) / (frecuencias + normalizador)
    return puntuaciones

puntuaciones_bm25 = np.vstack([calcular_bm25(q["tokens"]) for q in consultas])
rankings_bm25 = []
for qi, consulta in enumerate(consultas):
    tabla = construir_tabla_ranking(puntuaciones_bm25[qi], qi)
    rankings_bm25.append(tabla)
    display(Markdown(f"**{consulta['id']}: {consulta['texto']} — BM25**"))
    display(tabla)

comparacion_rankings = []
for qi, consulta in enumerate(consultas):
    top_tfidf = rankings_tfidf[qi]["Identificador"].tolist()
    top_bm25 = rankings_bm25[qi]["Identificador"].tolist()
    comparacion_rankings.append({"Consulta": consulta["id"], "Top-5 TF-IDF (IDs)": top_tfidf,
                                 "Top-5 BM25 (IDs)": top_bm25,
                                 "Documentos comunes": len(set(top_tfidf) & set(top_bm25)),
                                 "Cambia el primero": top_tfidf[0] != top_bm25[0]})
tabla_comparacion = pd.DataFrame(comparacion_rankings)
display(tabla_comparacion)
cambios = int(tabla_comparacion["Cambia el primero"].sum())
display(Markdown(f"El primer documento cambia en **{cambios} de 4 consultas**. "
                 "TF-IDF con coseno compara orientaciones; BM25 combina saturación de frecuencias "
                 "y corrección por longitud. Se comparan las posiciones y los documentos recuperados; "
                 "sus puntuaciones tienen escalas diferentes. Sin juicios de relevancia, los cambios "
                 "no permiten afirmar que un método sea mejor."))
sin_coincidencia = [(consultas[qi]["id"], ids[fila]) for (qi, fila), datos in fragmentos_guardados.items()
                    if datos["criterio"].startswith("Inicio")]
if sin_coincidencia:
    print("Fragmentos del inicio del cuerpo en alguno de los métodos (consulta, ID):", sin_coincidencia)


**Q1: alice rabbit wonderland — BM25**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,11,Alice's Adventures in Wonderland,12.449461,[Illustration]\n\n\n\n\nAlice’s Adventures in Wonderland\n\nby Lewis Carroll\n\nTHE MILLENNIUM FULCRUM EDITION 3.0\n\nContents\n\n CHAPTER I. Down the Rabbit-Hole\n CHAPTER II. The Pool o
1,2,157,Daddy-Long-Legs,8.950922,rella\nor Blue Beard or Robinson Crusoe or Jane Eyre or Alice in Wonderland or\na word of Rudyard Kipling. I didn't know that Henry the Eighth was\nmarried more than once or that She
2,3,12,Through the Looking-Glass,7.254963,[Illustration]\n\n\n\n\nTHROUGH THE LOOKING-GLASS\n\nAnd What Alice Found There\n\nBy Lewis Carroll\n\nThe Millennium Fulcrum Edition 1.7\n\n\n\n\nDRAMATIS PERSONÆ.\n(_As arranged before commenceme
3,4,130,Orthodoxy,6.918058,r and an idea of welcome. We need to be\nhappy in this wonderland without once being merely comfortable.\nIt is THIS achievement of my creed that I shall chiefly pursue in\nthese pag
4,5,152,Wild Justice,6.421204,"g,\nthe number of years in the profession and clients.\n\nAlice Stebbins was quite short. Her features gave her age\nas around fifty and holding. She dressed severely, in browns\nand"


**Q2: pirates treasure island — BM25**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,120,Treasure Island,6.673571,"TREASURE ISLAND\n\nby Robert Louis Stevenson\n\n\n\n\nTREASURE ISLAND\n\nTo S.L.O., an American gentleman in accordance with whose classic taste\nthe following narrative has been designed, i"
1,2,16,Peter Pan [Peter and Wendy],6.514953,"AY, COME AWAY!\n Chapter IV. THE FLIGHT\n Chapter V. THE ISLAND COME TRUE\n Chapter VI. THE LITTLE HOUSE\n Chapter VII. THE HOME UNDER THE GROUND\n Chapter VIII. THE MERMAIDS’ LAGOON\n C"
2,3,128,The Arabian Nights Entertainments,5.536136,"so she did not let me drown, but transported me to an\nisland. When the day dawned, she said to me,\n\n""When I saw you on the sea-shore I took a great fancy to you, and\nwished to try"
3,4,15,"Moby-Dick; or, The Whale",5.309227,"ideration of his guarding and protecting the seas from\npirates and robbers, is the right to _royal_ fish, which are whale\nand sturgeon. And these, when either thrown ashore or caug"
4,5,101,The Hacker Crackdown: Law and Disorder on the Electronic Frontier,5.258689,"roven to be shameless and persistent\nsoftware and data pirates. Universities do not merely\n""leak information"" but vigorously broadcast free thought.\n\nThis clash of values has been"


**Q3: love marriage elinor marianne — BM25**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,161,Sense and Sensibility,18.433471,"children added a relish to his existence.\n\nBy a former marriage, Mr. Henry Dashwood had one son: by his present\nlady, three daughters. The son, a steady respectable young man, was\n"
1,2,145,Middlemarch,8.299750,XXI.\n CHAPTER XXXII.\n CHAPTER XXXIII.\n\n BOOK IV. THREE LOVE PROBLEMS.\n CHAPTER XXXIV.\n CHAPTER XXXV.\n CHAPTER XXXVI.\n CHAPTER XXXVII.\n CHAPTER XXXVIII.\n CHAPTER XXXIX.\n CHAPTER XL.
2,3,138,George Sand: Some Aspects of Her Life and Writings,6.660572,(DE BOURGES)\n VI A CASE OF MATERNAL AFFECTION IN LOVE\n VII THE HUMANITARIAN DREAM\n VIII 1848\n IX THE 'BONNE DAME' OF NOHANT\n X THE GENIUS OF THE WRITER\n
3,4,180,The 1994 CIA World Factbook,2.694255,"arty, Mimi Stilling JAKOBSEN; Radical Liberal\n Party, Marianne JELVED; Christian People's Party, Jann SJURSEN; Common\n Course, Preben Moller HANSEN; Danish Workers' Party\nMember"
4,5,200,"The Project Gutenberg Encyclopedia, Volume 1 of 28",1.958530,"d overtake a family\nif a girl member should, after her marriage, ever set eyes\non her mother: hence the Ababda husband has to make his\nhome far from his wife's village. In the Mah"


**Q4: united states independence constitution — BM25**

,Posición,Identificador,Título,Puntuación,Fragmento
0,1,14,The 1990 CIA World Factbook,3.715501,"ally by the Central Intelligence\nAgency for the use of United States Government officials, and the style,\nformat, coverage, and content are designed to meet their specific\nrequirem"
1,2,180,The 1994 CIA World Factbook,3.713188,"ritania\nMauritius\nMayotte\nMexico\nMicronesia, Federated States of\nMidway Islands\nMoldova\nMonaco\nMongolia\nMontserrat\nMorocco\nMozambique\n\nN\nNamibia\nNauru\nNavassa Island\nNepal\nNetherla"
2,3,18,The Federalist Papers,3.636389,o. VI. Concerning Dangers from Dissensions Between the States\n FEDERALIST No. VII. The Same Subject Continued (Concerning Dangers from Dissensions Between the States)\n FEDERALIST N
3,4,200,"The Project Gutenberg Encyclopedia, Volume 1 of 28",3.423561,"ated.\nAlthough the text is in the public domain in the United States,\nthe original publisher still has a valid trademark in the\noriginal title of the encyclopedia. The original pu"
4,5,134,"Maria; Or, The Wrongs of Woman",3.300644,"ater part of which she hoarded, as her only chance for\nindependence, were much more considerable than she could reckon on\nobtaining any where else, were it possible that she, an ou"


,Consulta,Top-5 TF-IDF (IDs),Top-5 BM25 (IDs),Documentos comunes,Cambia el primero
0,Q1,"[11, 12, 133, 159, 19]","[11, 157, 12, 130, 152]",2,False
1,Q2,"[16, 120, 159, 180, 14]","[120, 16, 128, 15, 101]",2,True
2,Q3,"[161, 100, 138, 163, 145]","[161, 145, 138, 180, 200]",3,False
3,Q4,"[1, 2, 18, 117, 156]","[14, 180, 18, 200, 134]",1,True


El primer documento cambia en **2 de 4 consultas**. TF-IDF con coseno compara orientaciones; BM25 combina saturación de frecuencias y corrección por longitud. Se comparan las posiciones y los documentos recuperados; sus puntuaciones tienen escalas diferentes. Sin juicios de relevancia, los cambios no permiten afirmar que un método sea mejor.

Fragmentos del inicio del cuerpo en alguno de los métodos (consulta, ID): [('Q4', '117'), ('Q4', '156')]


### Verificación general

Las siguientes comprobaciones abarcan el corpus, las representaciones completas, las cuatro consultas, los rankings y la procedencia de los fragmentos. BM25 también se contrasta con el cálculo escalar sobre los contadores originales para todos los documentos.

In [12]:
# Se comprueban identidad, vocabulario y frecuencias de todos los documentos.
assert ids == IDS_ORIGINALES and len(set(ids)) == N == 100
assert len({doc["sha256"] for doc in documentos}) == N
assert vocabulario == sorted(vocabulario_corpus)
assert all(columnas[t] == i for i, t in enumerate(vocabulario))
assert matriz_tf.shape == matriz_tfidf.shape == matriz_sklearn.shape == (N, len(vocabulario))
assert np.issubdtype(matriz_tf.dtype, np.integer) and np.all(matriz_tf.data > 0)
for i, doc in enumerate(documentos):
    assert doc["tokens"] == tokenizar(doc["texto"])
    assert contadores[i] == Counter(doc["tokens"])
    fila = matriz_tf.getrow(i)
    assert dict(zip((vocabulario[j] for j in fila.indices), fila.data)) == contadores[i]
    assert doc["longitud"] == sum(contadores[i].values()) == longitudes[i]
    assert doc["unicos"] == len(contadores[i])
assert np.array_equal(np.asarray(matriz_tf.sum(axis=1)).ravel(), longitudes)
df_independiente = Counter(t for doc in documentos for t in set(doc["tokens"]))
assert np.array_equal(df, [df_independiente[t] for t in vocabulario])
assert np.all((df >= 1) & (df <= N))
assert np.allclose(idf, np.log(N / df) + 1, atol=TOLERANCIA, rtol=0)
assert np.allclose(idf[df == N], 1, atol=TOLERANCIA, rtol=0)
assert np.array_equal(matriz_tf.indptr, matriz_tfidf.indptr)
assert np.array_equal(matriz_tf.indices, matriz_tfidf.indices)
assert np.allclose(matriz_tfidf.data, matriz_tf.data * idf[matriz_tf.indices], atol=TOLERANCIA, rtol=0)
for ejemplo in ejemplos:
    i, j = ids.index(ejemplo["Documento"]), columnas[ejemplo["Término"]]
    assert np.isclose(matriz_tfidf[i, j], ejemplo["TF-IDF"], atol=TOLERANCIA, rtol=0)
assert diferencia_tfidf <= TOLERANCIA and diferencia_consultas <= TOLERANCIA
assert vectorizador.get_feature_names_out().tolist() == vocabulario
assert np.array_equal(idf, idf_antes_consultas)
assert np.array_equal(vectorizador.idf_, idf_sklearn_antes)
assert vectorizador.vocabulary_ == vocabulario_antes
assert all(q["tokens"] == tokenizar(q["texto"]) for q in consultas)
assert np.isfinite(similitudes_manual).all() and np.all(normas_q > 0)
assert np.allclose(similitudes_manual, similitudes_sklearn, atol=TOLERANCIA, rtol=0)

# Se verifica BM25 con frecuencias de Counter, sin usar la matriz de pesos TF-IDF.
assert np.isclose(longitud_media, sum(len(d["tokens"]) for d in documentos) / N)
assert np.allclose(idf_bm25, np.log(1 + (N - df + 0.5) / (df + 0.5)), atol=TOLERANCIA, rtol=0)
assert np.isfinite(puntuaciones_bm25).all() and np.all(puntuaciones_bm25 >= 0)
for qi, consulta in enumerate(consultas):
    terminos = sorted(set(consulta["tokens"]) & set(columnas))
    for i, frecuencias in enumerate(contadores):
        esperado = 0.0
        for t in terminos:
            f, df_t = frecuencias[t], df_independiente[t]
            idf_t = np.log1p((N - df_t + 0.5) / (df_t + 0.5))
            esperado += idf_t * f * (K1 + 1) / (f + K1 * (1 - B + B * len(documentos[i]["tokens"]) / longitud_media))
        assert np.isclose(puntuaciones_bm25[qi, i], esperado, atol=TOLERANCIA, rtol=0)
    for puntuaciones, tabla in [(similitudes_manual[qi], rankings_tfidf[qi]),
                               (puntuaciones_bm25[qi], rankings_bm25[qi])]:
        orden = ordenar_documentos(puntuaciones)
        assert np.all(np.diff(puntuaciones[orden]) <= 0)
        assert tabla["Identificador"].tolist() == [ids[i] for i in orden[:5]]
        assert tabla["Posición"].tolist() == [1, 2, 3, 4, 5]
        assert np.allclose(tabla["Puntuación"], puntuaciones[orden[:5]], atol=TOLERANCIA, rtol=0)
        for a, b in zip(orden, orden[1:]):
            if puntuaciones[a] == puntuaciones[b]:
                assert int(ids[a]) < int(ids[b])
assert ordenar_documentos(np.zeros(N)) == sorted(range(N), key=lambda i: int(ids[i]))
assert np.array_equal(calcular_bm25([]), np.zeros(N))
assert np.array_equal(calcular_bm25(tokenizar(termino_ausente)), np.zeros(N))

for (qi, i), datos in fragmentos_guardados.items():
    assert datos["fragmento"] == documentos[i]["texto"][datos["inicio"]:datos["fin"]]
    assert 0 < len(datos["fragmento"]) <= 180
    if datos["criterio"].startswith("Coincidencia"):
        assert set(tokenizar(datos["fragmento"])) & set(consultas[qi]["tokens"])

comprobaciones = ["Mismos documentos en todas las actividades", "Identificadores únicos",
                  "Vocabulario ordenado y columnas alineadas", "TF: conteos reales",
                  "DF: documentos, sin repetir apariciones", "IDF: ln(N/DF) + 1",
                  "TF-IDF: producto sin normalización", "Matrices equivalentes a scikit-learn",
                  "Consultas sin modificar IDF ni vocabulario", "Mismo preprocesamiento en consultas",
                  "Coseno explícito equivalente y normas cero", "Rankings ordenados y top-5 verificados",
                  "Fragmentos extraídos del texto original", "BM25 con estadísticas reales del corpus"]
display(pd.DataFrame({"Comprobación": comprobaciones, "Resultado": ["Correcto"] * len(comprobaciones)}))
print(f"Resumen: {N} documentos; {len(vocabulario):,} términos; matriz {matriz_tfidf.shape}; "
      f"4 consultas; diferencia TF-IDF {diferencia_tfidf:.12g}; diferencia coseno {diferencia_coseno:.12g}.")
print("Se verificaron los cuatro top-5 de TF-IDF y los cuatro de BM25.")


,Comprobación,Resultado
0,Mismos documentos en todas las actividades,Correcto
1,Identificadores únicos,Correcto
2,Vocabulario ordenado y columnas alineadas,Correcto
3,TF: conteos reales,Correcto
4,"DF: documentos, sin repetir apariciones",Correcto
5,IDF: ln(N/DF) + 1,Correcto
6,TF-IDF: producto sin normalización,Correcto
7,Matrices equivalentes a scikit-learn,Correcto
8,Consultas sin modificar IDF ni vocabulario,Correcto
9,Mismo preprocesamiento en consultas,Correcto


Resumen: 100 documentos; 191,063 términos; matriz (100, 191063); 4 consultas; diferencia TF-IDF 0; diferencia coseno 1.17267306976e-15.
Se verificaron los cuatro top-5 de TF-IDF y los cuatro de BM25.


**Reproducción y alcance.** Las salidas corresponden al corpus local indicado en la primera actividad. Para repetirlas se debe seleccionar el entorno con las dependencias mostradas, mantener los 100 TXT originales y ejecutar todas las celdas en orden. Si cambia su ubicación, se configura `RI_CORPUS` antes de iniciar el kernel. No se descargan documentos y no se modifica ningún notebook anterior. Los encabezados y licencias influyen en los pesos porque se conserva el criterio del trabajo anterior; los resultados miden coincidencia léxica y no sustituyen una evaluación humana de relevancia.

En Q4, los IDs 117 y 156 aparecen en el top-5 de TF-IDF aunque sus cuerpos tratan sobre una sinfonía y no contienen los términos buscados. Es un efecto observable de procesar también los encabezados y las licencias. Se conservan esos documentos para mantener exactamente el corpus del trabajo anterior, y sus fragmentos se identifican como tomados del inicio del cuerpo.